# Lecture 20: Savio Intro & Demo

AY 128; Apr. 2, 2026

# Agenda

- Setting up & Navigating Savio
- Training on Savio (GPU cluster)

## What is "Savio"?

From their website: "Savio is a Linux cluster with more than 600 nodes and 15,000 processor cores rated at more than 450 peak teraFLOPS." Essentially, a set of high performacne computers you can log into from anywhere with an internet connection to run your intensive code.

Great for:
- Code that takes too long to run. Can provide more computing power and offloads it from your computer.
- Jobs that need specific hardware; Some problems can be *much* faster on accelerated hardware like GPUs.

## Savio Cluster Basics
**Cluster Architecture**
* Login Nodes: Lightweight tasks (editing code, job submission).
* Compute Nodes: Heavy computations (request via SLURM).
* Partitions: Groups of nodes with specific resources (e.g., `savio2_1080ti` for GPU jobs).

**Storage Options**
* Home Directory: Small, for critical files.
* Scratch: High-speed storage for temporary job data.
* Condos: Purchased storage for large projects.

**Job Scheduling (SLURM)**
* SLURM manages resource allocation.
* Submit jobs via `sbatch job.sh`.
* Monitor jobs with `squeue -u $USER`.

## Using Savio: Step-by-Step
**Important Links**
* https://ucb-datalab.github.io/resources/savio/
    * Course specific documentation and distillation of the most relevant parts of the main documentation
* https://mybrc.brc.berkeley.edu/
    * Portal for managing your access to different computing projects
* https://docs-research-it.berkeley.edu/services/high-performance-computing/
    * Documentation for how to use the cluster
* https://ood.brc.berkeley.edu/
    * The On Demand portal: the simplest way to interface with the cluster

**Accessing Savio and File Transfer**
* Access, transfer files, edit files, and submit jobs from the On Demand portal

**Making a Job Script**
The required items for all job scripts is bolded. We will go over the main 
`SBATCH` options we use for our jobs.

* **Account** (`--account=ic_ay128f25`)
    * An allocation for the entire class
* **Node Partition (CPU)** (`--partition=savio2_htc`)
    * This is the "cheapest" node for single CPU cores (For the most part we 
    are not doing any parallel CPU computing)
* CPUs (`--cpus-per-task=1`)
* **Node Partition (GPU)** (`--partition=savio2_1080ti`)
    * This is the "cheapest" node with GPUs, more than enough compute for us
* GPU (`--gres=gpu:1`)
* CPUs (`--cpus-per-task=2`)
    * `savio2_1080ti` nodes need 2 CPU cores for each GPU requested
* **Time limit**: 10-minute runtime (`--time=00:10:00`)
    * The job will stop itself after this time limit is reached whether or not 
    the job is complete

## Training on Savio

We just trained a small MLP on tabular data right here in the previous notebook. But what happens when:
- The model is much larger (e.g., ResNet with millions of parameters)?
- The dataset is images instead of 8 numbers per sample?
- Training takes hours instead of seconds?

That's when we move to **GPU-accelerated computing** on a cluster like [Savio](https://docs-research-it.berkeley.edu/services/high-performance-computing/). The `savio/` directory in this repo has (almost) everything you need.

### What's in `savio/`?

```
savio/
├── examples/
│   ├── example_job.py          # Generic training script (works with any experiment)
│   ├── mnist/
│   │   └── experiment.py       # Simple fully-connected NN on handwritten digits
│   └── cifar10/
│       └── experiment.py       # ResNet-18 on natural images (10 classes)
└── job_scripts/
    ├── cpu_job.sh              # SLURM script for CPU training
    └── gpu_job.sh              # SLURM script for GPU training
```

The pattern is: `experiment.py` defines the model, data, loss, and optimizer. `example_job.py` handles the training loop, device selection (CPU vs GPU), and metric logging. The SLURM job scripts submit these to the cluster.

### The Experiments

#### MNIST (Handwritten Digits)
A simple fully-connected network (like our `NNClf` above, but for classification):
- Input: 28×28 grayscale images of digits 0–9
- Architecture: Flatten → Linear(784→128) → ReLU → Linear(128→64) → ReLU → Linear(64→10)
- Loss: CrossEntropyLoss (for classification, not regression)

<img src="../../savio/examples/mnist/data_example.png">

#### CIFAR-10 (Natural Images)
A ResNet-18 — a much deeper convolutional network:
- Input: 32×32 color images in 10 classes (airplane, car, bird, cat, deer, dog, frog, horse, ship, truck)
- Architecture: ResNet-18 with 18 layers, skip connections, ~11M parameters
- This is where GPUs really matter — training on CPU would take far too long

<img src="../../savio/examples/cifar10/data_example.png">


### CPU vs GPU

The key difference on Savio:

```bash
# CPU job (good for small models / debugging)
sbatch cpu_job.sh    # runs: python example_job.py --cpu --num_epochs=1

# GPU job (for real training)
sbatch gpu_job.sh    # runs: python example_job.py --gpu --num_epochs=10
```

PyTorch makes the CPU/GPU switch easy — `model.to(device)` and `tensor.to(device)` move computation to the right hardware. The `example_job.py` script handles this automatically based on the `--cpu` / `--gpu` flags.

We'll walk through running these on Savio in the demo!

## Demo Time!

First, we have to get set up:
- Check if our our cluster account is active.
- Log onto the cluster
- Setup the course environment

Let's train this model for 5 epochs 3 different ways:
1. Personal Computer on the CPU
    * Too slow for iterating designs, and 
2. Personal Computer on the GPU
    * Possible, but inconvienient
3. On Savio
    * Set it, and (hopefully) forget it